# components.session_list

> Session list component with virtual collection integration, keyboard navigation, and action modals.

Mirrors the structure of `cjm-transcript-workflow-management/components/document_list.py` with three key simplifications:

1. **No bulk selection** — sessions are single-row actions (Resume, Rename, Delete), no checkboxes
2. **Enricher-backed columns** — cell renderer reads from `item.enriched_fields[key]`, not raw attributes
3. **Active session indicator** — the current session row shows a success badge

In [ ]:
#| default_exp components.session_list

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Optional

from fasthtml.common import Div, Span, A, Button, Input, Label, Script, H3, Form, Dialog, P

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import (
    btn, btn_colors, btn_styles, btn_sizes
)
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.actions.modal import (
    modal, modal_box, modal_action, modal_backdrop,
)
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, min_h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Virtual collection
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef, CellRenderContext,
    VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.components.collection import render_virtual_collection

# Keyboard navigation
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_virtual_collection.keyboard.actions import (
    create_collection_focus_zone,
    create_collection_nav_actions,
    build_collection_url_map,
    apply_nav_sync,
)

# JS generators
from cjm_fasthtml_virtual_collection.js.scroll import generate_scroll_nav_js
from cjm_fasthtml_virtual_collection.js.scrollbar import generate_scrollbar_js
from cjm_fasthtml_virtual_collection.js.auto_fit import generate_auto_fit_js, auto_fit_callback_name

# Viewport fit
from cjm_fasthtml_viewport_fit.models import ViewportFitConfig
from cjm_fasthtml_viewport_fit.components import render_viewport_fit_script

# Local
from cjm_fasthtml_workflow_session_management.models import (
    EnrichedSessionSummary, SessionManagementUrls, ColumnSpec,
)
from cjm_fasthtml_workflow_session_management.html_ids import SessionManagerHtmlIds
from cjm_fasthtml_workflow_session_management.utils import format_relative_time, format_bytes
from cjm_fasthtml_workflow_session_management.components.helpers import (
    render_icon_button, render_empty_state, render_delete_modal,
    render_active_session_badge, DEBUG_SESSION_RENDER,
)

## Column Definitions

Fixed columns: label (resolved label + active badge) → status (current step) → updated (relative time) → size → actions. Host-supplied enricher columns are inserted between "status" and "updated" based on the `column_specs` argument.

Only the label column is sortable at the VC level — other columns are either host-derived (enricher) or backed by DB-level sorting via the service. The sort callback in `init_session_manager_routers()` re-fetches sorted items from the service rather than sorting the list in place.

In [ ]:
#| export
def build_session_columns(
    column_specs:Optional[List[ColumnSpec]]=None, # Host-supplied enricher column specs
) -> tuple: # Tuple of ColumnDef for the VC config
    """Build the column definitions for the session list.
    
    Produces a fixed column layout with host enricher columns inserted between
    status and updated columns. If `column_specs` is None or empty, only fixed
    columns are included.
    """
    specs = column_specs or []
    cols = [
        ColumnDef(key="label", header="Session", sortable=True),
        ColumnDef(key="status", header="Step", sortable=True),
    ]
    for spec in specs:
        cols.append(ColumnDef(
            key=spec.field,
            header=spec.header,
            sortable=False,
            header_cls=spec.width_class or "",
            cell_cls=spec.width_class or "",
        ))
    cols.extend([
        ColumnDef(key="updated", header="Updated", sortable=True),
        ColumnDef(key="size", header="Size", sortable=False),
        ColumnDef(key="actions", header="", sortable=False, header_cls=str(w("40"))),
    ])
    return tuple(cols)

In [ ]:
_cols_no_enr = build_session_columns()
assert [c.key for c in _cols_no_enr] == ["label", "status", "updated", "size", "actions"]

_specs = [ColumnSpec(field="sources", header="Sources"), ColumnSpec(field="segments", header="Segments")]
_cols_enr = build_session_columns(_specs)
assert [c.key for c in _cols_enr] == ["label", "status", "sources", "segments", "updated", "size", "actions"]
assert _cols_enr[0].sortable  # label
assert _cols_enr[1].sortable  # status
assert not _cols_enr[2].sortable  # enricher columns never sortable
print(f"Columns (with enricher): {[c.key for c in _cols_enr]}")

Columns (with enricher): ['label', 'status', 'sources', 'segments', 'updated', 'size', 'actions']


## Cell Renderer Factory

The cell renderer is built as a factory that captures the URLs, the active-session-ID getter, and the step-label resolver in a closure. The returned function dispatches on `ctx.column.key`:

- `label` → resolved label + optional active badge
- `status` → current step ID mapped to a human title via `get_step_title` (or raw ID if no mapper)
- `updated` → relative time ("2 hours ago")
- `size` → human-readable byte size
- `actions` → Resume / Rename / Delete icon buttons
- **Any other key** → lookup in `item.enriched_fields[key]` (the host enricher fields)

The fallback to `enriched_fields` for unknown keys means hosts can declare arbitrary enricher columns and they just work — the cell renderer doesn't need to know their names.

In [ ]:
#| export
def create_session_cell_renderer(
    urls:SessionManagementUrls, # URL bundle for action buttons
    get_active_session_id:Callable[[], str], # () -> currently active session ID
    get_step_title:Optional[Callable[[str], str]]=None, # Optional step ID -> human title mapper
) -> Callable: # render_cell(item, ctx) -> FT component
    """Create a cell renderer for the session list virtual collection."""
    def render_cell(
        item:EnrichedSessionSummary, # Enriched session at this row
        ctx:CellRenderContext, # Context with row_index, column, etc.
    ) -> Any: # FT component for the cell
        key = ctx.column.key
        sid = item.summary.session_id
        
        if key == "label":
            children = [Span(item.resolved_label, cls=str(font_weight.medium))]
            if sid == get_active_session_id():
                children.append(render_active_session_badge())
            return Div(*children, cls=combine_classes(flex_display, items.center, gap(2)))
        
        if key == "status":
            step_id = item.summary.current_step or ""
            title = get_step_title(step_id) if get_step_title and step_id else step_id
            return Span(title or "—", cls=str(font_size.sm))
        
        if key == "updated":
            return Span(
                format_relative_time(item.summary.updated_at),
                title=item.summary.updated_at or "",
                cls=str(font_size.sm),
            )
        
        if key == "size":
            return Span(format_bytes(item.summary.state_size_bytes), cls=str(font_size.sm))
        
        if key == "actions":
            safe_label = (item.resolved_label or "").replace("'", "\\'").replace('"', '\\"')
            return Div(
                render_icon_button(
                    "circle-play", "Resume",
                    size=str(btn_sizes.sm),
                    hx_post=urls.resume_session,
                    hx_vals=f'{{"session_id": "{sid}"}}',
                    hx_swap="none",
                    onclick="event.stopPropagation()",
                ),
                render_icon_button(
                    "square-pen", "Rename",
                    size=str(btn_sizes.sm),
                    onclick=(
                        f"event.stopPropagation(); "
                        f"prepareRenameModal('{sid}', '{safe_label}'); "
                        f"document.getElementById('{SessionManagerHtmlIds.RENAME_MODAL}').showModal()"
                    ),
                ),
                render_icon_button(
                    "trash-2", "Delete",
                    color=str(btn_colors.error),
                    size=str(btn_sizes.sm),
                    onclick=(
                        f"event.stopPropagation(); "
                        f"prepareDeleteModal('{sid}', '{safe_label}'); "
                        f"document.getElementById('{SessionManagerHtmlIds.DELETE_MODAL}').showModal()"
                    ),
                ),
                cls=combine_classes(flex_display, gap(1)),
            )
        
        # Fallback: enricher-backed column.
        value = item.enriched_fields.get(key, "—")
        return Span(value, cls=str(font_size.sm))
    
    return render_cell

In [ ]:
# Cell renderer dispatch test with a stub EnrichedSessionSummary.
from cjm_workflow_state.state_store import SessionSummary

_test_urls = SessionManagementUrls(
    management_page="/m/sessions/",
    list_sessions="/m/sessions/list",
    session_detail="/m/sessions/detail",
    create_session="/m/sessions/create",
    delete_session="/m/sessions/delete",
    rename_session="/m/sessions/rename",
    resume_session="/m/sessions/resume",
)

_test_item = EnrichedSessionSummary(
    summary=SessionSummary(
        flow_id="decomp", session_id="abc123",
        label=None, current_step="segmentation",
        created_at="2026-04-08 10:00:00",
        updated_at="2026-04-08 11:30:00",
        state_size_bytes=123456,
    ),
    resolved_label="My Session",
    enriched_fields={"sources": "3", "segments": "89"},
)

_step_titles = {"selection": "Selection", "segmentation": "Segment & Align", "verify": "Verify"}
_get_active = lambda: "abc123"  # this row IS active
_render = create_session_cell_renderer(_test_urls, _get_active, lambda sid: _step_titles.get(sid, sid))

_cols = build_session_columns([ColumnSpec(field="sources", header="Sources")])
_col_by_key = {c.key: c for c in _cols}

def _make_ctx(col):
    return CellRenderContext(row_index=0, column=col, total_items=1, is_cursor=False)

# Label column: should have the active badge (since get_active returns "abc123").
label_cell = _render(_test_item, _make_ctx(_col_by_key["label"]))
assert label_cell.tag == "div"
# First child is the label Span, second is the active badge Span.
assert label_cell.children[0].children[0] == "My Session"
assert len(label_cell.children) == 2  # label + active badge
assert "badge-success" in label_cell.children[1].attrs["class"]

# Not-active case: no badge.
_render2 = create_session_cell_renderer(_test_urls, lambda: "other-id", lambda sid: sid)
label_cell2 = _render2(_test_item, _make_ctx(_col_by_key["label"]))
assert len(label_cell2.children) == 1  # label only, no badge

# Status column: mapped via get_step_title.
status_cell = _render(_test_item, _make_ctx(_col_by_key["status"]))
assert status_cell.children[0] == "Segment & Align"

# Updated column: relative time.
updated_cell = _render(_test_item, _make_ctx(_col_by_key["updated"]))
# The relative time depends on current time; just confirm it's a Span with text.
assert updated_cell.tag == "span"

# Size column: formatted bytes.
size_cell = _render(_test_item, _make_ctx(_col_by_key["size"]))
assert "KB" in size_cell.children[0]  # 123456B → 120.6 KB

# Actions column: three icon buttons.
actions_cell = _render(_test_item, _make_ctx(_col_by_key["actions"]))
assert actions_cell.tag == "div"
assert len(actions_cell.children) == 3  # resume, rename, delete
resume_btn, rename_btn, delete_btn = actions_cell.children
assert resume_btn.attrs.get("hx-post") == "/m/sessions/resume"
assert "prepareRenameModal" in rename_btn.attrs["onclick"]
assert "prepareDeleteModal" in delete_btn.attrs["onclick"]
assert "btn-error" in delete_btn.attrs["class"]

# Enricher fallback: "sources" column resolves via item.enriched_fields["sources"].
sources_cell = _render(_test_item, _make_ctx(_col_by_key["sources"]))
assert sources_cell.children[0] == "3"

# Unknown key falls through to enricher lookup with "—" default.
_mystery_col = ColumnDef(key="mystery", header="?", sortable=False)
mystery_cell = _render(_test_item, _make_ctx(_mystery_col))
assert mystery_cell.children[0] == "—"

print("Cell renderer dispatch OK")

Cell renderer dispatch OK


## Toolbar

The session-list toolbar is simpler than the document-list toolbar — no bulk selection, no delete-selected. It contains a session count on the left and a "New Session" button on the right.

In [ ]:
#| export
def render_session_toolbar(
    urls:SessionManagementUrls, # URL bundle for action routes
    total_count:int=0, # Total number of sessions in the list
) -> Any: # Toolbar element
    """Render the session list toolbar with session count and New Session button."""
    count_label = "1 session" if total_count == 1 else f"{total_count} sessions"
    return Div(
        # Left: session count
        Span(
            count_label,
            cls=combine_classes(font_size.sm, text_dui.base_content.opacity(70)),
        ),
        # Right: New Session button
        Button(
            lucide_icon("plus", size=4),
            "New Session",
            type="button",  # Prevent form-submit hijack inside any wrapping form.
            cls=combine_classes(
                btn, btn_colors.primary, btn_sizes.sm,
                flex_display, items.center, gap(1),
            ),
            id=SessionManagerHtmlIds.NEW_SESSION_BTN,
            hx_post=urls.create_session,
            hx_swap="none",
        ),
        id=SessionManagerHtmlIds.TOOLBAR,
        cls=combine_classes(flex_display, justify.between, items.center, m.b(2)),
    )

In [ ]:
_tb = render_session_toolbar(_test_urls, total_count=3)
assert _tb.tag == "div"
assert _tb.attrs["id"] == SessionManagerHtmlIds.TOOLBAR
count_span, new_btn = _tb.children
assert count_span.children[0] == "3 sessions"
assert new_btn.attrs.get("hx-post") == _test_urls.create_session
assert "btn-primary" in new_btn.attrs["class"]
assert new_btn.attrs["type"] == "button"  # Prevent form-submit hijack.

# Singular.
_tb1 = render_session_toolbar(_test_urls, total_count=1)
assert _tb1.children[0].children[0] == "1 session"
print("Toolbar OK")

## Rename Modal

New modal specific to the session manager — HTML5 `<dialog>` with a text input for the new label. The actual session ID and current label are injected by `prepareRenameModal()` (defined in `render_list_scripts()`), which populates the input value and sets the confirm button's `hx-vals`.

In [ ]:
#| export
def render_rename_modal(
    urls:SessionManagementUrls, # URL bundle for rename route
    modal_id:str=SessionManagerHtmlIds.RENAME_MODAL, # Dialog element ID
    input_id:str=SessionManagerHtmlIds.RENAME_INPUT, # Input element ID
) -> Any: # Dialog element
    """Render the rename session modal using HTML5 dialog with backdrop click-to-close."""
    return Dialog(
        Div(
            H3("Rename Session", cls=combine_classes(font_size.lg, font_weight.bold)),
            Div(
                Input(
                    type="text",
                    id=input_id,
                    name="label",
                    placeholder="Session label",
                    cls=combine_classes(text_input, w.full),
                ),
                cls=m.y(4),
            ),
            Div(
                Form(
                    Button("Cancel", cls=str(btn), formmethod="dialog"),
                    method="dialog",
                ),
                Button(
                    lucide_icon("check", size=4),
                    "Rename",
                    type="button",  # Prevent form-submit hijack inside any wrapping form.
                    cls=combine_classes(
                        btn, btn_colors.primary,
                        flex_display, items.center, gap(1),
                    ),
                    data_rename_confirm="true",
                    hx_post=urls.rename_session,
                    hx_swap="none",
                    onclick=f"document.getElementById('{modal_id}').close()",
                ),
                cls=combine_classes(modal_action, flex_display, gap(2)),
            ),
            cls=str(modal_box),
        ),
        # Backdrop — plain unstyled button per DaisyUI v5 pattern. Clicking
        # anywhere outside the modal_box submits this form (method="dialog")
        # which closes the dialog natively.
        Form(Button("close"), method="dialog", cls=str(modal_backdrop)),
        id=modal_id,
        cls=str(modal),
    )

In [ ]:
_rm = render_rename_modal(_test_urls)
assert _rm.tag == "dialog"
assert _rm.attrs["id"] == SessionManagerHtmlIds.RENAME_MODAL

# Dialog has two children: modal box and backdrop form.
assert len(_rm.children) == 2
box, backdrop = _rm.children
assert "modal-box" in box.attrs["class"]

# Backdrop form: closes on click.
assert backdrop.tag == "form"
assert backdrop.attrs["method"] == "dialog"
assert "modal-backdrop" in backdrop.attrs["class"]

# Inner structure: header, body (with input), action row.
assert box.children[0].tag == "h3"
input_el = box.children[1].children[0]
assert input_el.tag == "input"
assert input_el.attrs["id"] == SessionManagerHtmlIds.RENAME_INPUT

# Confirm button: has rename data attribute AND type="button".
action_row = box.children[2]
confirm_btn = action_row.children[1]
assert confirm_btn.attrs.get("data-rename-confirm") == "true"
assert confirm_btn.attrs.get("hx-post") == _test_urls.rename_session
assert confirm_btn.attrs["type"] == "button"
print("Rename modal OK (with backdrop + type=button)")

## Client-Side Scripts

Two JavaScript functions injected into the page:

- `prepareDeleteModal(sessionId, label)` — populates the delete modal body with confirmation text and sets the confirm button's `hx-vals` to the target session ID.
- `prepareRenameModal(sessionId, currentLabel)` — populates the rename input with the current label and sets the confirm button's `hx-vals` to the target session ID. On confirm, the input's current value is sent as the new label.

In [ ]:
#| export
def render_list_scripts(
    urls:SessionManagementUrls, # URL bundle for action routes
) -> Any: # Script element
    """Render client-side JavaScript for delete and rename modal management."""
    faded = combine_classes(font_size.sm, text_dui.base_content.opacity(60), m.t(2))
    return Script(f"""
    function prepareDeleteModal(sessionId, label) {{
        var body = document.getElementById('{SessionManagerHtmlIds.DELETE_MODAL_BODY}');
        if (body) {{
            body.innerHTML = '<p>Permanently delete session "' + label + '"?</p>'
                + '<p class="{faded}">This clears all progress for this session. This action cannot be undone.</p>';
        }}
        var confirmBtn = document.querySelector('#{SessionManagerHtmlIds.DELETE_MODAL} [data-delete-confirm]');
        if (confirmBtn) {{
            confirmBtn.setAttribute('hx-post', '{urls.delete_session}');
            confirmBtn.setAttribute('hx-vals', JSON.stringify({{session_id: sessionId}}));
            confirmBtn.setAttribute('hx-swap', 'none');
            htmx.process(confirmBtn);
        }}
    }}

    function prepareRenameModal(sessionId, currentLabel) {{
        var input = document.getElementById('{SessionManagerHtmlIds.RENAME_INPUT}');
        if (input) {{
            input.value = currentLabel || '';
            setTimeout(function() {{ input.focus(); input.select(); }}, 50);
        }}
        var confirmBtn = document.querySelector('#{SessionManagerHtmlIds.RENAME_MODAL} [data-rename-confirm]');
        if (confirmBtn) {{
            // The label is read from the input on click via hx-include; session_id travels via hx-vals.
            confirmBtn.setAttribute('hx-vals', 'js:{{session_id: "' + sessionId + '", label: document.getElementById("{SessionManagerHtmlIds.RENAME_INPUT}").value}}');
            htmx.process(confirmBtn);
        }}
    }}
    """)

## Full List Renderer

Assembles everything: toolbar, virtual collection, alert area, delete modal, rename modal, keyboard system, and scripts. Returns the root list container with the standard `SessionManagerHtmlIds.SESSION_LIST` id so it can be targeted by OOB swaps.

In [ ]:
#| export
def render_session_list(
    items:list, # Current session list (List[EnrichedSessionSummary])
    vc_config:VirtualCollectionConfig, # VC configuration
    vc_state:VirtualCollectionState, # VC state
    vc_ids:VirtualCollectionHtmlIds, # VC HTML IDs
    vc_btn_ids:VirtualCollectionButtonIds, # VC button IDs
    vc_urls:VirtualCollectionUrls, # VC route URLs
    mgmt_urls:SessionManagementUrls, # Session manager URLs
    render_cell:Callable, # Cell renderer callback
) -> Any: # Complete session list component
    """Render the session list with virtual collection, keyboard nav, and modals."""
    if DEBUG_SESSION_RENDER:
        print(f"[RENDER] session_list: {len(items)} sessions")
    
    # Empty state short-circuit.
    if not items:
        return Div(
            render_session_toolbar(mgmt_urls, total_count=0),
            render_empty_state(
                message="No sessions found.",
                detail="Click New Session to start your first workflow.",
            ),
            # Persistent scripts + modals so toolbar's New Session still works.
            render_list_scripts(mgmt_urls),
            render_delete_modal(
                modal_id=SessionManagerHtmlIds.DELETE_MODAL,
                body_id=SessionManagerHtmlIds.DELETE_MODAL_BODY,
                title="Delete Session?",
                confirm_attrs={
                    "data_delete_confirm": "true",
                    "onclick": f"document.getElementById('{SessionManagerHtmlIds.DELETE_MODAL}').close()",
                },
            ),
            render_rename_modal(mgmt_urls),
            id=SessionManagerHtmlIds.SESSION_LIST,
            cls=combine_classes(flex_display, flex_direction.col, grow(1), min_h(0)),
        )
    
    # Keyboard system.
    zone = create_collection_focus_zone(vc_ids)
    nav_actions = create_collection_nav_actions(zone.id, vc_btn_ids)
    manager = ZoneManager(zones=(zone,), actions=nav_actions)
    url_map = build_collection_url_map(vc_btn_ids, vc_urls)
    target_map = {btn_id: f"#{vc_ids.rows}" for btn_id in url_map}
    swap_map = {btn_id: "none" for btn_id in url_map}
    kb_system = render_keyboard_system(
        manager,
        url_map=url_map,
        target_map=target_map,
        swap_map=swap_map,
    )
    apply_nav_sync(kb_system, vc_ids)
    
    # Viewport fit for auto-sizing.
    vf_config = ViewportFitConfig(
        namespace=vc_config.prefix,
        target_id=vc_ids.wrapper,
        resize_callback=auto_fit_callback_name(vc_config),
        enable_htmx_settle=False,
    )
    
    # JavaScript for VC navigation.
    js_code = "\n".join([
        generate_scroll_nav_js(vc_ids, vc_btn_ids),
        generate_scrollbar_js(vc_ids, vc_urls),
        generate_auto_fit_js(vc_ids, vc_config, vc_urls, total_items=len(items)),
    ])
    
    return Div(
        render_session_toolbar(mgmt_urls, total_count=len(items)),
        Div(
            render_virtual_collection(
                items=items,
                config=vc_config,
                state=vc_state,
                ids=vc_ids,
                urls=vc_urls,
                render_cell=render_cell,
                render_empty=lambda: render_empty_state(
                    message="No sessions found.",
                    detail="Click New Session to start your first workflow.",
                ),
            ),
            cls=combine_classes(
                flex_display, flex_direction.col, grow(1), min_h(0), overflow.hidden,
            ),
        ),
        Div(id=SessionManagerHtmlIds.ALERT_AREA),
        render_delete_modal(
            modal_id=SessionManagerHtmlIds.DELETE_MODAL,
            body_id=SessionManagerHtmlIds.DELETE_MODAL_BODY,
            title="Delete Session?",
            confirm_attrs={
                "data_delete_confirm": "true",
                "onclick": f"document.getElementById('{SessionManagerHtmlIds.DELETE_MODAL}').close()",
            },
        ),
        render_rename_modal(mgmt_urls),
        kb_system.script,
        kb_system.hidden_inputs,
        kb_system.action_buttons,
        render_list_scripts(mgmt_urls),
        Script(js_code),
        render_viewport_fit_script(vf_config),
        id=SessionManagerHtmlIds.SESSION_LIST,
        cls=combine_classes(flex_display, flex_direction.col, grow(1), min_h(0)),
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()